# Bilan carbone Datarmor

Estimation des émissions liées à l'usage de Datarmor (Ifremer) par les utilisateurs affiliés à MARBEC (groupes unix `emeX`, `marbecmd`).

La logique de calcul est dans [datarmor.py](datarmor.py), testée par [tests/test_datarmor.py](tests/test_datarmor.py) (`python -m pytest tests/`).

```
CPU (s) par user      ──Σ, /3600──► heures-cœur
                                       │ × kWh/heure-cœur  (= E_machine / H_machine, Grafana)
                                       │ × facteur d'overhead (PUE, seulement si non inclus dans E_machine)
                                       ▼
                                      kWh ── × kgCO2e/kWh (ADEME) ──► tCO2e
```

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import datarmor as dm

## Paramètres

⚠️ Les statistiques machine (`MACHINE`) proviennent des dashboards Grafana **2023** (du 01/01/2023 au 31/12/2023) :

- https://grafana.ifremer.fr/d/000000022/datarmor-diris?orgId=1&from=1672527600000&to=1704063599000
- https://grafana.ifremer.fr/d/000000038/datarmor-pbsacctsql-rapport-exploit?orgId=1&from=1672527600000&to=1704063599000

Il faut les remplacer par les valeurs de l'année `YEAR` dès qu'elles sont disponibles. Les trois termes de `core_hours` (en kh) restent à documenter : files ou partitions ?

In [ ]:
YEAR = 2025
# Anonymised pbsacct report (logins -> user01, user02..., ranked by Ucput_s)
CPU_FILE = f"data-datarmor/marbec-datarmor-{YEAR}_anonyme.csv"

MACHINE = dm.MachineStats(
    year=2023,                                   # TODO: update with YEAR values
    energy_kwh=1.64e6,                           # DIRIS, total electricity
    core_hours=(2003 + 47780 + 17196) * 1000,    # pbsacct, kh -> h
)

KGCO2E_PER_KWH = 0.052   # ADEME Base Empreinte, French average mix (check year)
PUE = 1.42               # former "FCYP" (LOCEAN); likely already in DIRIS energy

if MACHINE.year != YEAR:
    print(f"WARNING: machine statistics are for {MACHINE.year}, CPU data for {YEAR}")
print(f"Mean energy per core-hour: {MACHINE.kwh_per_core_hour:.4f} kWh "
      f"(~{1000 * MACHINE.kwh_per_core_hour:.1f} W per core)")

## Heures CPU par utilisateur

Pour 2025, l'Ifremer a fourni un rapport `pbsacct` avec **une ligne par utilisateur et des totaux annuels** (pas de détail mensuel), contrairement aux fichiers mensuels de 2024.

Le fichier brut (`data-datarmor/marbec-2025.txt`) contient les logins et n'est pas versionné. Le notebook lit la version anonymisée, produite ainsi :

```python
report = dm.read_pbsacct_report("data-datarmor/marbec-2025.txt")
anon, mapping = dm.anonymize_users(report, sort_by="Ucput_s")
anon.to_csv("data-datarmor/marbec-datarmor-2025_anonyme.csv")
```

Colonnes utilisées :

- `Ucput_s` : temps CPU réellement consommé (secondes), **utilisé pour le bilan** (même grandeur que les fichiers des années précédentes) ;
- `EFWALCoreUsed` : vraisemblablement cœurs réservés × durée (cœur-secondes), donné à titre de comparaison. Cette interprétation reste à confirmer avec l'Ifremer.

In [ ]:
report = dm.read_pbsacct_report(CPU_FILE)
n_months = dm.EXPECTED_MONTHS  # annual totals: the whole year is covered

summary = report[["count(jobid)", "Ucput_s", "EFWALCoreUsed"]].sum()
pd.Series({
    "users": len(report),
    "jobs": int(summary["count(jobid)"]),
    "core-hours used (Ucput_s)": round(summary["Ucput_s"] / dm.SECONDS_PER_HOUR),
    "core-hours reserved (EFWALCoreUsed)": round(summary["EFWALCoreUsed"] / dm.SECONDS_PER_HOUR),
}).to_frame(f"{YEAR}")

In [ ]:
hours = dm.cpu_hours_by_user(report[["Ucput_s"]])

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(hours))
ax.bar(x, hours)
ax.set_xticks(x)
ax.set_xticklabels(hours.index, rotation=45, ha="right")
ax.set_xlim(x.min() - 0.5, x.max() + 0.5)
ax.set_ylabel("CPU time (core-hours)")
ax.set_title(f"Datarmor CPU usage per MARBEC user, {YEAR} ({n_months} months)")
fig.tight_layout()
fig.savefig(f"datarmor_usage_{YEAR}.png")

total_seconds = float(report["Ucput_s"].sum())
total_hours = total_seconds / dm.SECONDS_PER_HOUR
print(f"Total: {total_hours:,.0f} core-hours "
      f"({100 * total_hours / MACHINE.core_hours:.1f} % of the {MACHINE.year} machine total)")

## Empreinte carbone : scénarios

Deux incertitudes principales :

1. **Facteur 1.42 (ex-« FCYP »)**. Il est appliqué à des kWh, donc il joue le rôle d'un PUE. Si la mesure DIRIS inclut déjà le refroidissement, il est compté deux fois. ~24 W par cœur, c'est nettement plus que le TDP par cœur des processeurs, ce qui suggère que l'énergie mesurée inclut déjà l'overhead. La fabrication du matériel **n'est pas** couverte par ce facteur et devrait être estimée à part (Berthoud et al. 2020, hal-02549565).
2. **Mois manquants**. Le scénario « extrapolé » multiplie le total par 12 / nombre de mois disponibles.

In [5]:
extrapolation = dm.EXPECTED_MONTHS / n_months
scenarios = {
    "no overhead":                 (1.0, 1.0),
    f"PUE {PUE}":                  (PUE, 1.0),
    "no overhead, extrapolated":   (1.0, extrapolation),
    f"PUE {PUE}, extrapolated":    (PUE, extrapolation),
}
rows = []
for name, (overhead, scale) in scenarios.items():
    seconds = total_seconds * scale
    rows.append({
        "scenario": name,
        "core-hours": seconds / dm.SECONDS_PER_HOUR,
        "energy (kWh)": dm.energy_kwh(seconds, MACHINE, overhead),
        "tCO2e": dm.footprint_tco2e(seconds, MACHINE, KGCO2E_PER_KWH, overhead),
    })
results = pd.DataFrame(rows).set_index("scenario")
results.style.format({"core-hours": "{:,.0f}", "energy (kWh)": "{:,.0f}", "tCO2e": "{:.1f}"})

,core-hours,energy (kWh),tCO2e
scenario,,,
no overhead,"8,819,095","215,938",11.2
PUE 1.42,"8,819,095","306,632",15.9
"no overhead, extrapolated","9,620,831","235,569",12.2
"PUE 1.42, extrapolated","9,620,831","334,508",17.4


## Comparaison avec Irene (TGCC)

D'après [notes-irenes.txt](notes-irenes.txt) : 4 000 000 h ≈ 4 tCO2e, soit environ 1e-6 tCO2e par heure CPU. Ce facteur couvre déjà toute la machine Irene, donc on n'applique pas de PUE supplémentaire. C'est un simple ordre de grandeur, car il s'agit d'une machine différente.

In [6]:
TCO2E_PER_CORE_HOUR_IRENE = 1e-6
irene = total_hours * TCO2E_PER_CORE_HOUR_IRENE
print(f"Irene factor: {irene:.1f} tCO2e | Datarmor (no overhead): "
      f"{results.loc['no overhead', 'tCO2e']:.1f} tCO2e")

Irene factor: 8.8 tCO2e | Datarmor (no overhead): 11.2 tCO2e


## Annexe : données 2023 (format mail)

Pour 2023, les données avaient été reçues par mail au format `YYYYMM:user : secondes`.

In [7]:
mail_2023 = dm.parse_mail_cpu_seconds("data-datarmor/mail-mickael-2023.txt")
by_user_2023 = mail_2023.groupby("user")["cpu_seconds"].sum().div(dm.SECONDS_PER_HOUR).rename("core-hours")
by_user_2023 = by_user_2023[by_user_2023 > 0].sort_values(ascending=False)
print(f"2023: {by_user_2023.sum():,.0f} core-hours, "
      f"months present: {sorted(mail_2023['month'].unique())}")
by_user_2023.head(10)

2023: 6,600,768 core-hours, months present: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


user
ldalaut     1.455311e+06
mlengaig    9.063674e+05
abrunel     8.359495e+05
vdaniell    8.130546e+05
jrault      7.256522e+05
nbarrier    7.155548e+05
adupaix     2.405749e+05
lbald       2.217120e+05
mdouille    1.378105e+05
cabello     1.344551e+05
Name: core-hours, dtype: float64